<a href="https://colab.research.google.com/github/toobaidrees/Tooba_Idrees_Data_Analytics/blob/main/Phase%202%20task%205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Task 5: Interactive Business Dashboard in Streamlit

**Objective:** Build an interactive Streamlit dashboard for analyzing Sales, Profit, and Segment-wise performance from the Global Superstore dataset.

**Pipeline:**
1. 📦 Install dependencies & tunnel setup
2. 📂 Create the Global Superstore dataset (inline)
3. 🧹 Data cleaning & preparation
4. 🚀 Write & launch the Streamlit app
5. 🔗 Access via public URL (using localtunnel)

---
> ⚠️ **Colab Note:** Streamlit runs as a local server. We use **localtunnel** to expose it publicly so you can open it in your browser.

## 📦 Cell 1 – Install Dependencies

In [1]:
!pip install streamlit plotly localtunnel -q
print('✅ streamlit, plotly, localtunnel installed')

ERROR: Ignored the following versions that require a different python version: 0.55.2 Requires-Python <3.5
ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel
✅ streamlit, plotly, localtunnel installed


## 📂 Cell 2 – Create the Global Superstore Dataset (Inline)

> The full Kaggle dataset has ~50k rows. We embed a representative 200-row sample so **no file upload or API key is needed**.
>
> To use the real dataset: upload `Global Superstore.csv` from Kaggle and uncomment Option B below.

In [2]:
import pandas as pd
import numpy as np
import io, os, random
random.seed(42)
np.random.seed(42)

# ── Option B: Upload your own CSV ─────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # upload Global Superstore.csv
# df = pd.read_csv(list(uploaded.keys())[0], encoding='latin-1')

# ── Option A (DEFAULT): Generate a realistic synthetic dataset ────────────
regions     = ['West', 'East', 'Central', 'South']
categories  = ['Technology', 'Furniture', 'Office Supplies']
sub_cats    = {
    'Technology'     : ['Phones', 'Accessories', 'Copiers', 'Machines'],
    'Furniture'      : ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
    'Office Supplies': ['Binders', 'Paper', 'Storage', 'Labels', 'Envelopes'],
}
segments    = ['Consumer', 'Corporate', 'Home Office']
countries   = ['United States'] * 6 + ['United Kingdom', 'Australia', 'Germany', 'France']
ship_modes  = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
customers   = [
    'Sean Miller','Tamara Chand','Raymond Buch','Tom Ashbrook','Adrian Barton',
    'Ken Lonsdale','Sanjit Chand','Hunter Lopez','Bill Eplett','Chloris Kastensmidt',
    'Steven Cartwright','Christopher Conant','Greg Tran','Lena Radford','Pete Kriz',
    'Lari Sotelo','Jonathan Doherty','Odella Nelson','Edward Hooks','Jane Waco',
]

rows = []
for i in range(1, 501):
    cat     = random.choice(categories)
    sub     = random.choice(sub_cats[cat])
    region  = random.choice(regions)
    sales   = round(random.uniform(10, 5000), 2)
    disc    = round(random.choice([0, 0, 0, 0.1, 0.2, 0.3, 0.4, 0.5]), 2)
    margin  = {'Technology': 0.18, 'Furniture': 0.04, 'Office Supplies': 0.22}[cat]
    profit  = round(sales * margin * (1 - disc * 1.5) + random.uniform(-50, 50), 2)
    qty     = random.randint(1, 14)
    rows.append({
        'Order ID'       : f'CA-{2020+i//200}-{100000+i}',
        'Order Date'     : pd.Timestamp('2021-01-01') + pd.Timedelta(days=random.randint(0, 1460)),
        'Ship Date'      : pd.Timestamp('2021-01-05') + pd.Timedelta(days=random.randint(0, 1464)),
        'Ship Mode'      : random.choice(ship_modes),
        'Customer Name'  : random.choice(customers),
        'Segment'        : random.choice(segments),
        'Country'        : random.choice(countries),
        'Region'         : region,
        'Category'       : cat,
        'Sub-Category'   : sub,
        'Sales'          : sales,
        'Quantity'       : qty,
        'Discount'       : disc,
        'Profit'         : profit,
    })

df = pd.DataFrame(rows)
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month

# Save for Streamlit to load
df.to_csv('superstore.csv', index=False)

print(f'✅ Dataset ready: {df.shape[0]} rows × {df.shape[1]} cols')
print(f"   Sales  : ${df['Sales'].sum():,.0f}")
print(f"   Profit : ${df['Profit'].sum():,.0f}")
df.head()

✅ Dataset ready: 500 rows × 16 cols
   Sales  : $1,273,280
   Profit : $131,458


,Order ID,Order Date,Ship Date,Ship Mode,Customer Name,Segment,Country,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Year,Month
0,CA-2020-100001,2021-07-29,2024-10-21,Standard Class,Edward Hooks,Corporate,United States,West,Office Supplies,Binders,3710.34,12,0.1,666.15,2021,7
1,CA-2020-100002,2024-08-23,2024-12-11,Same Day,Hunter Lopez,Corporate,France,East,Technology,Phones,1170.98,12,0.0,216.90,2024,8
2,CA-2020-100003,2022-11-21,2021-08-02,Standard Class,Greg Tran,Consumer,United States,East,Furniture,Chairs,3493.72,4,0.3,54.65,2022,11
3,CA-2020-100004,2021-06-11,2024-02-09,First Class,Jane Waco,Corporate,France,West,Furniture,Bookcases,3651.36,7,0.0,193.37,2021,6
4,CA-2020-100005,2022-04-22,2021-07-30,Same Day,Bill Eplett,Corporate,United States,West,Technology,Phones,3309.70,14,0.2,465.54,2022,4


## 🧹 Cell 3 – Data Cleaning & Preparation Preview

In [3]:
print('── Shape ────────────────────────────────')
print(df.shape)

print('\n── Missing Values ──────────────────────')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else 'None ✅')

print('\n── Dtypes ──────────────────────────────')
print(df.dtypes)

print('\n── Summary Stats ───────────────────────')
display(df[['Sales','Profit','Quantity','Discount']].describe().round(2))

print('\n── Unique Filter Values ────────────────')
print('Regions    :', df['Region'].unique().tolist())
print('Categories :', df['Category'].unique().tolist())
print('Sub-Cats   :', df['Sub-Category'].unique().tolist())

── Shape ────────────────────────────────
(500, 16)

── Missing Values ──────────────────────
None ✅

── Dtypes ──────────────────────────────
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer Name            object
Segment                  object
Country                  object
Region                   object
Category                 object
Sub-Category             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
Year                      int32
Month                     int32
dtype: object

── Summary Stats ───────────────────────


,Sales,Profit,Quantity,Discount
count,500.00,500.00,500.00,500.00
mean,2546.56,262.92,7.76,0.20
std,1442.23,254.33,4.09,0.18
min,31.64,-41.59,1.00,0.00
25%,1354.58,69.89,4.00,0.00
50%,2600.73,169.62,8.00,0.20
75%,3755.72,409.48,11.00,0.40
max,4997.87,1130.41,14.00,0.50



── Unique Filter Values ────────────────
Regions    : ['West', 'East', 'Central', 'South']
Categories : ['Office Supplies', 'Technology', 'Furniture']
Sub-Cats   : ['Binders', 'Phones', 'Chairs', 'Bookcases', 'Copiers', 'Machines', 'Accessories', 'Storage', 'Paper', 'Envelopes', 'Furnishings', 'Labels', 'Tables']


## 🖥️ Cell 4 – Write the Streamlit App

This writes `app.py` to disk. The dashboard includes:
- **Sidebar filters**: Region, Category, Sub-Category, Year
- **KPI cards**: Total Sales, Total Profit, Profit Margin, Total Orders
- **Charts**: Monthly Sales Trend, Sales by Category, Profit by Region, Top 5 Customers, Segment Pie, Discount vs Profit scatter

In [4]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Page config ──────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Global Superstore Dashboard",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Custom CSS ───────────────────────────────────────────────────────────
st.markdown("""
<style>
    .main { background-color: #f0f2f6; }
    .kpi-card {
        background: white;
        border-radius: 12px;
        padding: 20px 24px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        border-left: 5px solid;
        margin-bottom: 8px;
    }
    .kpi-label { font-size: 13px; color: #666; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px; }
    .kpi-value { font-size: 28px; font-weight: 800; margin: 4px 0; }
    .kpi-delta { font-size: 12px; color: #888; }
    .section-title { font-size: 18px; font-weight: 700; color: #2d2d2d; margin: 24px 0 12px; border-bottom: 2px solid #e0e0e0; padding-bottom: 8px; }
</style>
""", unsafe_allow_html=True)

# ── Load data ────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("superstore.csv", parse_dates=["Order Date", "Ship Date"])
    df["Year"]  = df["Order Date"].dt.year
    df["Month"] = df["Order Date"].dt.month
    df["Month Name"] = df["Order Date"].dt.strftime("%b")
    df["Profit Margin %"] = (df["Profit"] / df["Sales"] * 100).round(2)
    return df

df = load_data()

# ── Sidebar filters ──────────────────────────────────────────────────────
st.sidebar.image("https://img.icons8.com/color/96/combo-chart--v2.png", width=60)
st.sidebar.title("🔍 Filters")
st.sidebar.markdown("---")

years = sorted(df["Year"].unique())
sel_years = st.sidebar.multiselect("📅 Year", years, default=years)

regions = ["All"] + sorted(df["Region"].unique())
sel_region = st.sidebar.selectbox("🌍 Region", regions)

categories = ["All"] + sorted(df["Category"].unique())
sel_cat = st.sidebar.selectbox("📦 Category", categories)

if sel_cat != "All":
    sub_cats = ["All"] + sorted(df[df["Category"] == sel_cat]["Sub-Category"].unique())
else:
    sub_cats = ["All"] + sorted(df["Sub-Category"].unique())
sel_sub = st.sidebar.selectbox("🏷️ Sub-Category", sub_cats)

segments = ["All"] + sorted(df["Segment"].unique())
sel_seg = st.sidebar.selectbox("👥 Segment", segments)

# ── Apply filters ────────────────────────────────────────────────────────
fdf = df[df["Year"].isin(sel_years)]
if sel_region != "All": fdf = fdf[fdf["Region"]       == sel_region]
if sel_cat    != "All": fdf = fdf[fdf["Category"]     == sel_cat]
if sel_sub    != "All": fdf = fdf[fdf["Sub-Category"] == sel_sub]
if sel_seg    != "All": fdf = fdf[fdf["Segment"]      == sel_seg]

st.sidebar.markdown("---")
st.sidebar.metric("Filtered Rows", f"{len(fdf):,}")

# ── Header ───────────────────────────────────────────────────────────────
st.markdown("## 📊 Global Superstore Business Dashboard")
st.markdown(f"Showing **{len(fdf):,}** orders after filters")
st.markdown("---")

# ── KPI Cards ────────────────────────────────────────────────────────────
total_sales   = fdf["Sales"].sum()
total_profit  = fdf["Profit"].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0
total_orders  = fdf["Order ID"].nunique()
avg_order_val = total_sales / total_orders if total_orders else 0

c1, c2, c3, c4 = st.columns(4)
kpi_data = [
    (c1, "💰 Total Sales",    f"${total_sales:,.0f}",      "#4ECDC4", "Revenue from all orders"),
    (c2, "📈 Total Profit",   f"${total_profit:,.0f}",     "#FF6B6B" if total_profit < 0 else "#6C5CE7", "Net profit after costs"),
    (c3, "📊 Profit Margin",  f"{profit_margin:.1f}%",     "#FFE66D", "Profit as % of sales"),
    (c4, "🛒 Total Orders",   f"{total_orders:,}",          "#A29BFE", f"Avg. ${avg_order_val:,.0f} / order"),
]
for col, label, value, color, delta in kpi_data:
    col.markdown(f"""
    <div class="kpi-card" style="border-left-color:{color}">
        <div class="kpi-label">{label}</div>
        <div class="kpi-value" style="color:{color}">{value}</div>
        <div class="kpi-delta">{delta}</div>
    </div>""", unsafe_allow_html=True)

st.markdown("")

# ── Row 1: Monthly Sales Trend + Sales by Category ───────────────────────
st.markdown('<div class="section-title">📅 Sales & Profit Trends</div>', unsafe_allow_html=True)
r1c1, r1c2 = st.columns([2, 1])

with r1c1:
    monthly = (fdf.groupby(["Year", "Month", "Month Name"])
                  .agg(Sales=("Sales","sum"), Profit=("Profit","sum"))
                  .reset_index()
                  .sort_values(["Year","Month"]))
    monthly["Period"] = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=monthly["Period"], y=monthly["Sales"],
                         name="Sales", marker_color="#4ECDC4", opacity=0.85), secondary_y=False)
    fig.add_trace(go.Scatter(x=monthly["Period"], y=monthly["Profit"],
                             name="Profit", line=dict(color="#FF6B6B", width=2.5),
                             mode="lines+markers", marker_size=5), secondary_y=True)
    fig.update_layout(title="Monthly Sales & Profit", template="plotly_white",
                      height=340, legend=dict(orientation="h", y=1.12),
                      margin=dict(t=60, b=20, l=20, r=20))
    fig.update_xaxes(tickangle=45)
    fig.update_yaxes(title_text="Sales ($)", secondary_y=False)
    fig.update_yaxes(title_text="Profit ($)", secondary_y=True)
    st.plotly_chart(fig, use_container_width=True)

with r1c2:
    cat_data = fdf.groupby("Category")[["Sales","Profit"]].sum().reset_index()
    fig = px.bar(cat_data, x="Category", y=["Sales","Profit"],
                 barmode="group", title="Sales & Profit by Category",
                 color_discrete_sequence=["#6C5CE7","#FFE66D"],
                 template="plotly_white", height=340)
    fig.update_layout(legend_title="", margin=dict(t=60, b=20, l=20, r=20))
    st.plotly_chart(fig, use_container_width=True)

# ── Row 2: Region Profit + Segment Pie ───────────────────────────────────
st.markdown('<div class="section-title">🌍 Region & Segment Analysis</div>', unsafe_allow_html=True)
r2c1, r2c2 = st.columns([1, 1])

with r2c1:
    reg_data = fdf.groupby("Region")[["Sales","Profit"]].sum().reset_index().sort_values("Sales", ascending=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(y=reg_data["Region"], x=reg_data["Sales"],
                         name="Sales", orientation="h", marker_color="#4ECDC4", opacity=0.9))
    fig.add_trace(go.Bar(y=reg_data["Region"], x=reg_data["Profit"],
                         name="Profit", orientation="h", marker_color="#A29BFE", opacity=0.9))
    fig.update_layout(title="Sales & Profit by Region", barmode="group",
                      template="plotly_white", height=320,
                      margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

with r2c2:
    seg_data = fdf.groupby("Segment")["Sales"].sum().reset_index()
    fig = px.pie(seg_data, names="Segment", values="Sales",
                 title="Sales by Customer Segment",
                 color_discrete_sequence=["#FF6B6B","#4ECDC4","#FFE66D"],
                 hole=0.45, template="plotly_white", height=320)
    fig.update_traces(textposition="inside", textinfo="percent+label",
                      pull=[0.04]*3)
    fig.update_layout(margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

# ── Row 3: Top 5 Customers + Sub-Category Performance ────────────────────
st.markdown('<div class="section-title">🏆 Top Customers & Sub-Category Performance</div>', unsafe_allow_html=True)
r3c1, r3c2 = st.columns([1, 1])

with r3c1:
    top_cust = (fdf.groupby("Customer Name")[["Sales","Profit"]]
                   .sum().reset_index()
                   .sort_values("Sales", ascending=False).head(5))
    fig = go.Figure()
    fig.add_trace(go.Bar(y=top_cust["Customer Name"][::-1],
                         x=top_cust["Sales"][::-1],
                         name="Sales", orientation="h",
                         marker_color="#6C5CE7", opacity=0.9))
    fig.add_trace(go.Bar(y=top_cust["Customer Name"][::-1],
                         x=top_cust["Profit"][::-1],
                         name="Profit", orientation="h",
                         marker_color="#FF6B6B", opacity=0.9))
    fig.update_layout(title="🏅 Top 5 Customers by Sales",
                      barmode="group", template="plotly_white",
                      height=320, margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

with r3c2:
    sub_data = (fdf.groupby("Sub-Category")[["Sales","Profit"]]
                   .sum().reset_index()
                   .sort_values("Sales", ascending=True).tail(8))
    fig = px.bar(sub_data, y="Sub-Category", x=["Sales","Profit"],
                 orientation="h", barmode="group",
                 title="Top Sub-Categories by Sales",
                 color_discrete_sequence=["#4ECDC4","#A29BFE"],
                 template="plotly_white", height=320)
    fig.update_layout(margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

# ── Row 4: Discount vs Profit scatter + Ship Mode ────────────────────────
st.markdown('<div class="section-title">🔬 Deep Dive</div>', unsafe_allow_html=True)
r4c1, r4c2 = st.columns([3, 2])

with r4c1:
    fig = px.scatter(fdf, x="Discount", y="Profit", color="Category",
                     size="Sales", hover_data=["Customer Name","Sub-Category","Region"],
                     title="Discount vs Profit (bubble size = Sales)",
                     color_discrete_sequence=["#FF6B6B","#4ECDC4","#FFE66D"],
                     template="plotly_white", height=340,
                     opacity=0.7)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Break-even")
    fig.update_layout(margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

with r4c2:
    ship_data = fdf.groupby("Ship Mode")["Sales"].sum().reset_index().sort_values("Sales")
    fig = px.funnel(ship_data, x="Sales", y="Ship Mode",
                    title="Sales by Shipping Mode",
                    color_discrete_sequence=["#6C5CE7"],
                    template="plotly_white", height=340)
    fig.update_layout(margin=dict(t=50,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

# ── Data Table ───────────────────────────────────────────────────────────
st.markdown('<div class="section-title">🗃️ Raw Data Explorer</div>', unsafe_allow_html=True)
with st.expander("📋 Click to view / download filtered data"):
    st.dataframe(fdf.sort_values("Sales", ascending=False)
                    .reset_index(drop=True),
                 use_container_width=True, height=300)
    csv = fdf.to_csv(index=False).encode()
    st.download_button("⬇️ Download CSV", csv, "filtered_superstore.csv", "text/csv")

# ── Footer ───────────────────────────────────────────────────────────────
st.markdown("---")
st.markdown(
    "<p style=\'text-align:center;color:#aaa;font-size:13px\'>"
    "📊 Global Superstore Dashboard &nbsp;|&nbsp; Built with Streamlit &amp; Plotly</p>",
    unsafe_allow_html=True
)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ app.py written successfully')
print('   Lines:', app_code.count('\n'))

✅ app.py written successfully
   Lines: 244


In [9]:
!pkill -f streamlit
try:
    from pyngrok import ngrok
    ngrok.kill()
except: pass
print('✅ Streamlit server stopped')

✅ Streamlit server stopped
